# Kelly vs. SBF vs. risk-neutral gamblers

Same compounding gamble as `a_risky_gamble.ipynb`: each round, the underlying asset return is $R = 2$ or $R = 0.25$, each w.p. $\tfrac{1}{2}$. Each player bets a fraction $f$ of wealth each round, so wealth evolves as
$$W_{t+1} \;=\; W_t \bigl(1 + f\,(R - 1)\bigr).$$
We simulate $N = 10{,}000$ players for each of three strategies, all taken from `Kelly_etc.ipynb`:

- **Kelly** ($f^* = 1/6 \approx 0.1667$) — the log-utility / $\gamma = 1$ optimum.
- **SBF** ($f^* \approx 0.2226$) — CRRA optimum at $\gamma = 0.75$, $K = 16$, on this gamble.
- **Risk neutral** ($f = 1$) — maximizes arithmetic $E[W_T]$ by going all-in each round; this is exactly the original `a_risky_gamble.ipynb` simulation.

Here we just play these strategies forward and see where the players end up.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(544)

In [2]:
initial_investment = 1000
T = 100         # rounds per player
N = 10_000      # players per strategy
returns = np.array([2.0, 0.25])

f_kelly = 1 / 6
f_sbf   = 0.2226    # CRRA optimum at gamma=0.75 on the {2, 0.25} gamble
f_rn    = 1.0       # risk-neutral: all-in each round

In [3]:
# Vectorized Monte Carlo: each row is one player's T draws of the underlying asset.
draws = rng.choice(returns, size=(N, T), p=[0.5, 0.5])

kelly_multipliers = 1 + f_kelly * (draws - 1)
sbf_multipliers   = 1 + f_sbf   * (draws - 1)
rn_multipliers    = 1 + f_rn    * (draws - 1)   # equals draws itself when f=1

kelly_final = initial_investment * np.prod(kelly_multipliers, axis=1)
sbf_final   = initial_investment * np.prod(sbf_multipliers,   axis=1)
rn_final    = initial_investment * np.prod(rn_multipliers,    axis=1)

## Expected final wealth: theory vs. simulation

Returns are i.i.d., so $E[W_T] = W_0 \cdot (E[\text{multiplier}])^T$. The arithmetic expected per-round multiplier under fraction $f$ is
$$E[1 + f(R-1)] \;=\; 1 + f\bigl(E[R] - 1\bigr) \;=\; 1 + 0.125\,f.$$

In [4]:
E_R = 0.5 * 2.0 + 0.5 * 0.25

def theoretical_E_WT(f):
    return initial_investment * (1 + f * (E_R - 1)) ** T

ev_table = pd.DataFrame({
    "Strategy":           ["Kelly (f = 1/6)",
                           "SBF (f = 0.2226)",
                           "Risk neutral (f = 1)"],
    "Theoretical E[W_T]": [f"${theoretical_E_WT(f_kelly):,.2f}",
                           f"${theoretical_E_WT(f_sbf):,.2f}",
                           f"${theoretical_E_WT(f_rn):,.2f}"],
    "Simulated E[W_T]":   [f"${kelly_final.mean():,.2f}",
                           f"${sbf_final.mean():,.2f}",
                           f"${rn_final.mean():,.2f}"],
})
print(ev_table.to_string(index=False))

            Strategy Theoretical E[W_T] Simulated E[W_T]
     Kelly (f = 1/6)          $7,861.12        $7,551.32
    SBF (f = 0.2226)         $15,556.71       $14,268.57
Risk neutral (f = 1)    $130,392,389.71            $2.18


## Percentiles of the wealth distribution

In [5]:
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99, 99.9, 99.99]

table = pd.DataFrame({
    "Percentile":            percentiles,
    "Kelly wealth":          [f"${v:,.2f}" for v in np.percentile(kelly_final, percentiles)],
    "SBF wealth":            [f"${v:,.2f}" for v in np.percentile(sbf_final,   percentiles)],
    "Risk-neutral wealth":   [f"${v:,.2f}" for v in np.percentile(rn_final,    percentiles)],
})
print(table.to_string(index=False))

 Percentile Kelly wealth    SBF wealth Risk-neutral wealth
       1.00      $118.42        $36.73               $0.00
       5.00      $280.69       $116.11               $0.00
      10.00      $499.01       $250.09               $0.00
      25.00    $1,182.84       $790.56               $0.00
      50.00    $2,803.77     $2,499.04               $0.00
      75.00    $6,645.97     $7,899.74               $0.00
      90.00   $15,753.41    $24,971.98               $0.00
      95.00   $28,006.06    $53,787.28               $0.00
      99.00   $66,384.74   $170,027.65               $0.01
      99.90  $209,808.55   $788,810.80              $31.25
      99.99  $373,005.41 $1,699,103.38           $2,001.40


## Top 10 final wealth values among the 10,000 players (each strategy)

In [6]:
kelly_top10 = np.sort(kelly_final)[::-1][:10]
sbf_top10   = np.sort(sbf_final)[::-1][:10]
rn_top10    = np.sort(rn_final)[::-1][:10]

top10_table = pd.DataFrame({
    "Rank":                 np.arange(1, 11),
    "Kelly wealth":         [f"${v:,.2f}" for v in kelly_top10],
    "SBF wealth":           [f"${v:,.2f}" for v in sbf_top10],
    "Risk-neutral wealth":  [f"${v:,.2f}" for v in rn_top10],
})
print(top10_table.to_string(index=False))

 Rank Kelly wealth    SBF wealth Risk-neutral wealth
    1  $497,323.97 $2,493,519.78          $16,000.00
    2  $372,992.98 $1,699,023.93           $2,000.00
    3  $372,992.98 $1,699,023.93           $2,000.00
    4  $279,744.73 $1,157,673.71             $250.00
    5  $279,744.73 $1,157,673.71             $250.00
    6  $279,744.73 $1,157,673.71             $250.00
    7  $279,744.73 $1,157,673.71             $250.00
    8  $279,744.73 $1,157,673.71             $250.00
    9  $279,744.73 $1,157,673.71             $250.00
   10  $209,808.55   $788,810.80              $31.25


**Takeaway.** All three strategies share the same per-round positive arithmetic EV, but the distributions could not be more different.

- **Risk-neutral** ($f = 1$) maximizes $E[W_T]$ — theoretical mean \$130M — yet simulated mean is *\$2.18*. Almost every player goes broke; the entire arithmetic mass lives in tail events that don't appear in 10,000 trials.
- **Kelly** ($f = 1/6$) leaves most of the arithmetic EV on the table (theoretical mean \$7.9k vs the risk-neutral \$130M) and instead delivers a robust outcome: median player triples wealth, even the 1st percentile keeps \$118.
- **SBF** ($f \approx 0.223$) sits between them. Median wealth (\$2,499) is nearly the same as Kelly's, but the right tail is much heavier: SBF's top player ends at \$2.5M vs Kelly's \$497k. Above the 75th percentile SBF dominates Kelly; below it Kelly dominates SBF.

This is the same lesson `a_risky_gamble.ipynb` made starkly: *expected wealth* and *what almost any actual player experiences* pull in opposite directions when wealth compounds. Kelly is the unique fraction that maximizes typical wealth growth.